# SQuAD 2.0 Dataset Exploration

This notebook explores the SQuAD 2.0 (Stanford Question Answering Dataset) to understand:
- Dataset structure and statistics
- Distribution of answerable vs unanswerable questions
- Question, context, and answer length distributions
- Sample questions and answers

**Date:** November 2025  
**Dataset:** SQuAD 2.0

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

print('Imports successful!')

## 1. Load Dataset

In [ ]:
# Load development set
data_dir = Path('../data/squad')
dev_file = data_dir / 'dev-v2.0.json'

with open(dev_file, 'r', encoding='utf-8') as f:
    squad_data = json.load(f)

print(f"Dataset version: {squad_data['version']}")
print(f"Number of articles: {len(squad_data['data'])}")

## 2. Extract Questions and Statistics

In [ ]:
# Extract all questions with metadata
questions_data = []

for article in squad_data['data']:
    title = article['title']
    for paragraph in article['paragraphs']:
        context = paragraph['context']
        context_length = len(context)
        
        for qa in paragraph['qas']:
            question_data = {
                'id': qa['id'],
                'title': title,
                'question': qa['question'],
                'question_length': len(qa['question']),
                'context': context,
                'context_length': context_length,
                'is_impossible': qa['is_impossible'],
                'answer': qa['answers'][0]['text'] if qa['answers'] else None,
                'answer_length': len(qa['answers'][0]['text']) if qa['answers'] else 0,
                'answer_start': qa['answers'][0]['answer_start'] if qa['answers'] else None
            }
            questions_data.append(question_data)

# Create DataFrame
df = pd.DataFrame(questions_data)

print(f"Total questions: {len(df)}")
print(f"\nFirst few rows:")
df.head()

## 3. Basic Statistics

In [ ]:
# Overall statistics
print("=" * 60)
print("SQUAD 2.0 STATISTICS (Dev Set)")
print("=" * 60)

total_questions = len(df)
answerable = df[~df['is_impossible']].shape[0]
unanswerable = df[df['is_impossible']].shape[0]

print(f"\nTotal Questions: {total_questions}")
print(f"Answerable: {answerable} ({answerable/total_questions*100:.1f}%)")
print(f"Unanswerable: {unanswerable} ({unanswerable/total_questions*100:.1f}%)")

print(f"\nUnique Articles: {df['title'].nunique()}")
print(f"Unique Contexts: {df['context'].nunique()}")

print(f"\nQuestion Length:")
print(f"  Mean: {df['question_length'].mean():.1f} characters")
print(f"  Median: {df['question_length'].median():.1f} characters")
print(f"  Min: {df['question_length'].min()} characters")
print(f"  Max: {df['question_length'].max()} characters")

print(f"\nContext Length:")
print(f"  Mean: {df['context_length'].mean():.1f} characters")
print(f"  Median: {df['context_length'].median():.1f} characters")
print(f"  Min: {df['context_length'].min()} characters")
print(f"  Max: {df['context_length'].max()} characters")

# Answer statistics (only for answerable questions)
answerable_df = df[~df['is_impossible']]
if len(answerable_df) > 0:
    print(f"\nAnswer Length (answerable only):")
    print(f"  Mean: {answerable_df['answer_length'].mean():.1f} characters")
    print(f"  Median: {answerable_df['answer_length'].median():.1f} characters")
    print(f"  Min: {answerable_df['answer_length'].min()} characters")
    print(f"  Max: {answerable_df['answer_length'].max()} characters")

## 4. Visualizations

In [ ]:
# Answerable vs Unanswerable Distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Pie chart
labels = ['Answerable', 'Unanswerable']
sizes = [answerable, unanswerable]
colors = ['#2ecc71', '#e74c3c']
explode = (0.05, 0)

axes[0].pie(sizes, explode=explode, labels=labels, colors=colors, autopct='%1.1f%%',
            shadow=True, startangle=90)
axes[0].set_title('Question Answerability Distribution', fontsize=14, fontweight='bold')

# Bar chart
axes[1].bar(labels, sizes, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_ylabel('Number of Questions', fontsize=12)
axes[1].set_title('Question Counts by Type', fontsize=14, fontweight='bold')
axes[1].grid(axis='y', alpha=0.3)

for i, v in enumerate(sizes):
    axes[1].text(i, v + 2, str(v), ha='center', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Length Distributions
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Question length distribution
axes[0, 0].hist(df['question_length'], bins=30, color='skyblue', edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['question_length'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["question_length"].mean():.1f}')
axes[0, 0].set_xlabel('Question Length (characters)', fontsize=11)
axes[0, 0].set_ylabel('Frequency', fontsize=11)
axes[0, 0].set_title('Question Length Distribution', fontsize=12, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(alpha=0.3)

# Context length distribution
axes[0, 1].hist(df['context_length'], bins=30, color='lightcoral', edgecolor='black', alpha=0.7)
axes[0, 1].axvline(df['context_length'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["context_length"].mean():.1f}')
axes[0, 1].set_xlabel('Context Length (characters)', fontsize=11)
axes[0, 1].set_ylabel('Frequency', fontsize=11)
axes[0, 1].set_title('Context Length Distribution', fontsize=12, fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(alpha=0.3)

# Answer length distribution (answerable only)
if len(answerable_df) > 0:
    axes[1, 0].hist(answerable_df['answer_length'], bins=30, color='lightgreen', edgecolor='black', alpha=0.7)
    axes[1, 0].axvline(answerable_df['answer_length'].mean(), color='red', linestyle='--', linewidth=2, 
                       label=f'Mean: {answerable_df["answer_length"].mean():.1f}')
    axes[1, 0].set_xlabel('Answer Length (characters)', fontsize=11)
    axes[1, 0].set_ylabel('Frequency', fontsize=11)
    axes[1, 0].set_title('Answer Length Distribution (Answerable Questions)', fontsize=12, fontweight='bold')
    axes[1, 0].legend()
    axes[1, 0].grid(alpha=0.3)

# Box plots comparing answerable vs unanswerable
box_data = [df[~df['is_impossible']]['question_length'], 
            df[df['is_impossible']]['question_length']]
axes[1, 1].boxplot(box_data, labels=['Answerable', 'Unanswerable'], patch_artist=True,
                   boxprops=dict(facecolor='lightyellow', alpha=0.7),
                   medianprops=dict(color='red', linewidth=2))
axes[1, 1].set_ylabel('Question Length (characters)', fontsize=11)
axes[1, 1].set_title('Question Length by Answerability', fontsize=12, fontweight='bold')
axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Sample Questions

In [ ]:
# Display sample answerable questions
print("=" * 80)
print("SAMPLE ANSWERABLE QUESTIONS")
print("=" * 80)

answerable_samples = df[~df['is_impossible']].sample(min(3, len(df[~df['is_impossible']])))

for idx, row in answerable_samples.iterrows():
    print(f"\n[Article: {row['title']}]")
    print(f"Context: {row['context'][:200]}...")
    print(f"\nQuestion: {row['question']}")
    print(f"Answer: {row['answer']}")
    print("-" * 80)

In [ ]:
# Display sample unanswerable questions
print("=" * 80)
print("SAMPLE UNANSWERABLE QUESTIONS")
print("=" * 80)

if len(df[df['is_impossible']]) > 0:
    unanswerable_samples = df[df['is_impossible']].sample(min(3, len(df[df['is_impossible']])))
    
    for idx, row in unanswerable_samples.iterrows():
        print(f"\n[Article: {row['title']}]")
        print(f"Context: {row['context'][:200]}...")
        print(f"\nQuestion: {row['question']}")
        print(f"Answer: [UNANSWERABLE]")
        print("-" * 80)
else:
    print("No unanswerable questions in sample.")

## 6. Topic Distribution

In [ ]:
# Questions per article/topic
topic_counts = df['title'].value_counts().head(15)

plt.figure(figsize=(14, 6))
topic_counts.plot(kind='barh', color='steelblue', edgecolor='black')
plt.xlabel('Number of Questions', fontsize=12)
plt.ylabel('Article Title', fontsize=12)
plt.title('Top 15 Articles by Question Count', fontsize=14, fontweight='bold')
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\nTotal unique articles: {df['title'].nunique()}")
print(f"Average questions per article: {len(df) / df['title'].nunique():.1f}")

## 7. Key Insights

**Summary of SQuAD 2.0 Dataset:**

1. **Answerability**: The dataset includes both answerable and unanswerable questions, making it more realistic and challenging

2. **Question Characteristics**: 
   - Questions are typically concise (40-50 characters)
   - Contexts are longer passages (500-800 characters)
   - Answers tend to be short spans within the context

3. **Challenge**: Models must not only extract answers but also recognize when no answer exists

4. **Diversity**: Dataset covers various topics from multiple Wikipedia articles

**Next Steps:**
- Compare with Wikipedia-generated Q&A dataset
- Evaluate LLM performance on clean vs messy data
- Test answerability detection capabilities

## 8. Save Processed Data

In [ ]:
# Save DataFrame for later use
output_dir = Path('../data/processed')
output_dir.mkdir(parents=True, exist_ok=True)

df.to_csv(output_dir / 'squad_dev_processed.csv', index=False)
print(f"✓ Saved processed data to: {output_dir / 'squad_dev_processed.csv'}")

# Save statistics summary
stats_summary = {
    'total_questions': int(len(df)),
    'answerable': int(answerable),
    'unanswerable': int(unanswerable),
    'unique_articles': int(df['title'].nunique()),
    'avg_question_length': float(df['question_length'].mean()),
    'avg_context_length': float(df['context_length'].mean()),
    'avg_answer_length': float(answerable_df['answer_length'].mean()) if len(answerable_df) > 0 else 0
}

with open(output_dir / 'squad_stats_summary.json', 'w') as f:
    json.dump(stats_summary, f, indent=2)
print(f"✓ Saved statistics summary to: {output_dir / 'squad_stats_summary.json'}")